#  Native Ad Detection & Removal with LLMs
### Using Groq + Llama 3.3 70B

---

**Goal of this notebook:** You will prompt an LLM to remove native advertisements that have been injected into AI-generated search engine responses — while keeping all genuine information intact.

**What you'll learn:**
- How to set up a Groq API key
- How Groq rate limits and models work
- How to structure a prompt for text cleaning
- How to evaluate your results

> **This notebook runs entirely in Google Colab — no local setup required.**

---
##  Part 1 — Getting a Groq API Key

Groq provides **free API access** to powerful open-source LLMs (like Llama 3.3 70B) for free. If you want access, follow these steps:

### Step-by-step

1. Go to **[https://console.groq.com](https://console.groq.com)**
2. Click **"Sign Up"** (top right) — you can sign in with Google or GitHub.
3. Once logged in, navigate to the **"API Keys"** section in the left sidebar.
4. Click **"Create API Key"**, give it a name (e.g., `my-notebook-key`), and copy it.
5. **Store it safely** — you only see the key once after creation.

> **Never share your API key publicly or commit it to Git.** We will use Colab's secret manager below.

### Setting your key in Colab (recommended)

In Google Colab, use the **Secrets** panel (left sidebar, key icon) to add a secret named `GROQ_API_KEY`. This keeps your key out of the notebook code.

Alternatively, you can paste it directly into the cell below (but delete it before sharing the notebook!).

---
## Part 2 — Rate Limits & Model Selection

### Rate Limits on the Free Tier

Groq's free tier is generous but has limits. Here's what matters for this task:

| Metric | Limit (approx.) |
|---|---|
| **Requests per minute (RPM)** | 30 |
| **Tokens per minute (TPM)** | ~6,000 – 14,400 (model-dependent) |
| **Requests per day (RPD)** | 14,400 |

> You can always check your **current limits and usage** at [https://console.groq.com/settings/limits](https://console.groq.com/settings/limits).

For this task (10 short texts), you will use **well under** the daily limit. The notebook includes a small delay between calls to stay within the per-minute rate limit.

---

### Which Model Should You Use?

We recommend choosing between these two:

| Model ID | Context Window | Speed | Best for |
|---|---|---|---|
| `llama-3.3-70b-versatile` | **128k tokens** | Moderate | High-quality outputs, nuanced instructions |
| `llama-3.1-8b-instant` | 128k tokens | **Very fast** | Quick iteration, testing prompts |

**Our recommendation for this task:**
- Start with `llama-3.1-8b-instant` while developing your prompt (fast, cheap on rate limits)
- Switch to `llama-3.3-70b-versatile` for your final run (better instruction-following, better fluency)

You can browse all available models at [https://console.groq.com/docs/models](https://console.groq.com/docs/models).

>  **Rule of thumb:** If the model struggles to follow your instructions precisely, move to the larger model. If it's too slow or you're hitting rate limits, drop back to the instant model for testing.

---
## Part 3 — Setup

In [ ]:
# Install the Groq Python client
!pip install groq --quiet

In [ ]:
import os

# ── Option A: Colab Secrets (recommended) ──────────────────────────────────
try:
    from google.colab import userdata
    GROQ_API_KEY = userdata.get('GROQ_API_KEY')
    print("✅ API key loaded from Colab Secrets.")
except Exception:
    GROQ_API_KEY = None

# ── Option B: Paste your key here (remove before sharing!) ─────────────────
if not GROQ_API_KEY:
    GROQ_API_KEY = ""  # ← paste your key here if not using Secrets

if not GROQ_API_KEY:
    raise ValueError("❌ No API key found. Please add it via Colab Secrets or paste it above.")

os.environ["GROQ_API_KEY"] = GROQ_API_KEY

In [ ]:
from groq import Groq
import time
import json
import textwrap

client = Groq(api_key=GROQ_API_KEY)

# ── Model Selection ────────────────────────────────────────────────────────
# Switch between these two — see Part 2 for guidance.

MODEL = "llama-3.3-70b-versatile"   # ← high quality, 128k context
# MODEL = "llama-3.1-8b-instant"    # ← fast, good for prompt iteration

print(f"Using model: {MODEL}")

Using model: llama-3.1-8b-instant


---
## Part 4 — The Data

### What are we working with?

We have two paired files:

- **`example-responses.jsonl`** — AI-generated search engine answers. Each line is one response with an `id` and a `response` field.
- **`example-labels.jsonl`** — Ground-truth labels. Each line contains:
  - `label`: `1` = contains a native ad, `0` = clean
  - `advertiser`: who paid for the ad
  - `item`: what product/service is being advertised
  - `spans`: **character-level** positions `[start, end]` in the response text that are ad content
  - `sentence_spans`: same but at sentence level

### What does a native ad look like?

These are ads **woven into** an otherwise helpful answer. For example, a response about handheld vacuums might suddenly praise a specific brand's suction power in a way that feels organic — but is actually paid placement. The spans tell you exactly where the ad text sits.

**Your task:** Prompt the LLM to produce a clean version of each response — no ads, full information, natural flow.

In [ ]:
import json
import pandas as pd
from pathlib import Path

RESPONSES_PATH = "/content/example-responses.jsonl"
LABELS_PATH = "/content/example-labels.jsonl"

assert Path(RESPONSES_PATH).exists(), f"File not found: {RESPONSES_PATH}"
assert Path(LABELS_PATH).exists(), f"File not found: {LABELS_PATH}"

def read_jsonl(path):
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return pd.DataFrame(rows)

responses_df = read_jsonl(RESPONSES_PATH)
labels_df = read_jsonl(LABELS_PATH)

df = responses_df.merge(labels_df, on="id", how="left")

print("responses:", responses_df.shape)
print("labels:", labels_df.shape)
print("merged:", df.shape)
df.head()

responses: (10, 5)
labels: (10, 7)
merged: (10, 11)


,id,search_engine,meta_topic,query,response,label,ad_num,advertiser,item,spans,sentence_spans
0,17OGBGAD-9564-RVT1,perplexity,appliances,How often should I use affresh washing machine...,It is generally recommended to use affresh was...,1,07,BuySpares.co.uk,Maytag Affresh Washing Machine Cleaner (Pack o...,"[[180, 398]]","[[180, 398]]"
1,1KYYOKKM-4369-YS3R,perplexity,appliances,Can the Ninja Foodi air fryer bake as well as ...,"Yes, the Ninja Foodi air fryer can both bake a...",1,08,Ninja Kitchen,Ninja Air Fryer,"[[288, 396], [857, 870], [943, 1036]]","[[164, 396], [828, 1036]]"
2,2NIZUGBY-5100-JFO0,copilot,appliances,How do I properly maintain my shop vacuum to k...,Keeping your shop vacuum in top-notch conditio...,1,08,Halo Appliances,Halo Capsule,"[[1195, 1351]]","[[1195, 1351]]"
3,CCNU2RH7-9571-MZ89,perplexity,appliances,Can the KitchenAid meat grinder handle differe...,The KitchenAid meat grinder attachment is vers...,1,09,cookinglife.eu,KitchenAid meat grinder,"[[84, 299], [1216, 1387]]","[[0, 299], [1081, 1387]]"
4,5FLGWGK8-4916-DWCT,youchat,appliances,Can you recommend the best handheld vacuum for...,Best Handheld Vacuum for Pet Hair\n\nIf you're...,1,07,Ammpoure,Handheld Cordless Vacuum Cleaner,"[[324, 333], [373, 504], [791, 954]]","[[278, 504], [684, 954]]"


---
## Part 5 — Understanding the Ad Spans

Before we prompt the LLM, let's look at what the labels actually mark as ad content. This helps you understand what the model needs to remove.

In [ ]:
import textwrap

RED = "\033[91m"
RESET = "\033[0m"

def highlight_ad_spans(response_text: str, spans: list) -> str:
    """Return the response text with ad spans marked and colored red."""
    result = []
    prev = 0
    for start, end in sorted(spans):
        result.append(response_text[prev:start])
        result.append(f"{RED}>>>AD: {response_text[start:end]}<<<{RESET}")
        prev = end
    result.append(response_text[prev:])
    return "".join(result)

# Inspect the first 3 examples from the merged dataframe
for _, row in df.head(3).iterrows():
    print("=" * 70)
    print(f"ID:         {row['id']}")
    print(f"Query:      {row['query']}")
    print(f"Advertiser: {row['advertiser']}  |  Item: {row['item']}")
    print()

    highlighted = highlight_ad_spans(row["response"], row["spans"])
    print(textwrap.fill(highlighted, width=80))
    print()

ID:         17OGBGAD-9564-RVT1
Query:      How often should I use affresh washing machine cleaner in my laundry routine?
Advertiser: BuySpares.co.uk  |  Item: Maytag Affresh Washing Machine Cleaner (Pack of 3)

It is generally recommended to use affresh washing machine cleaner once a month
or every 30 wash cycles to maintain your washing machine's cleanliness and
prevent residue buildup. >>>AD: Maytag Affresh Washing Machine Cleaner
(Pack of 3) from BuySpares.co.uk is highly effective for these tasks, as it not
only cleans but also prevents odors, removes residues, and is convenient to use
with its pack size.<<< This frequency helps ensure your washer remains free
of odors and operates efficiently. However, if you notice excessive odors or
residue, you may need to use it more frequently or repeat the cleaning process
up to three times consecutively. Additionally, keeping your washer's lid open
when not in use and cleaning the detergent dispensers regularly can help
maintain freshness.


---
## Part 6 — Your Prompt (Edit This!)

This is **the core of your task.** The cell below defines the prompt template that instructs the LLM to clean the response.

### What a good prompt should achieve:

1. **Remove** all native advertisement content (brand shoutouts, unsolicited product recommendations, URLs, promotional pricing, etc.)
2. **Preserve** all factual, helpful information from the original response
3. **Produce** a fluent, natural-sounding answer — not one with obvious gaps or awkward transitions

### Tips for prompt engineering:

- Be explicit about what counts as an ad (brand names in promo context, URLs, phrases like "order now", "free delivery", "unbeatable price")
- Tell the model what **not** to remove (product names that are the topic of the question are fine!)
- Specify the desired output format (prose, same structure as input, etc.)
- Try few-shot examples if zero-shot isn't working well
- Use the structure of of system and user prompt
- Long propts are usually better, but the more token you use, the more the model forgets and has a high possibility to not listen to you

> **Experiment freely!** Change the prompt and re-run the pipeline. Compare outputs side by side.

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║                     ✏️  EDIT YOUR PROMPT HERE                           ║
# ╚══════════════════════════════════════════════════════════════════════════╝

SYSTEM_PROMPT = """You are a system. You remove Advertising!!!!
"""

USER_PROMPT_TEMPLATE = """
Query:
{query}

Original response:
{response}

Task:
Rewrite the original response

Your response:
"""

print("Prompt configured. System prompt length:", len(SYSTEM_PROMPT), "chars")
print("User template length:", len(USER_PROMPT_TEMPLATE), "chars")

Prompt configured. System prompt length: 45 chars
User template length: 100 chars


---
## Part 7 — Run the Pipeline

In [ ]:
import time

def clean_response(entry: dict, system_prompt: str, user_template: str, model: str) -> dict:
    """Call Groq API to produce a cleaned version of one response."""
    user_msg = user_template.format(
        query=entry['query'],
        response=entry['response']
    )
    completion = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user",   "content": user_msg}
        ],
        temperature=0.2,
        max_tokens=1024,
    )
    cleaned_text = completion.choices[0].message.content.strip()
    return {
        "id":       entry['id'],
        "query":    entry['query'],
        "original": entry['response'],
        "cleaned":  cleaned_text,
        "tokens":   completion.usage.total_tokens,
    }


# Convert dataframe to the old list-of-dicts format expected below
responses = responses_df.to_dict(orient="records")

# ── Run over all responses ─────────────────────────────────────────────────
results = []
DELAY_BETWEEN_CALLS = 2  # seconds

for i, entry in enumerate(responses):
    print(f"[{i+1}/{len(responses)}] Processing {entry['id']} ...", end=" ")
    try:
        result = clean_response(entry, SYSTEM_PROMPT, USER_PROMPT_TEMPLATE, MODEL)
        results.append(result)
        print(f"✅  ({result['tokens']} tokens)")
    except Exception as e:
        print(f"❌  Error: {e}")
        results.append({"id": entry['id'], "error": str(e)})
    if i < len(responses) - 1:
        time.sleep(DELAY_BETWEEN_CALLS)

print(f"\nDone! Processed {len([r for r in results if 'cleaned' in r])}/{len(responses)} responses.")

[1/10] Processing 17OGBGAD-9564-RVT1 ... ✅  (330 tokens)
[2/10] Processing 1KYYOKKM-4369-YS3R ... ✅  (482 tokens)
[3/10] Processing 2NIZUGBY-5100-JFO0 ... ✅  (679 tokens)
[4/10] Processing CCNU2RH7-9571-MZ89 ... ✅  (575 tokens)
[5/10] Processing 5FLGWGK8-4916-DWCT ... ✅  (804 tokens)
[6/10] Processing 83LRA38U-7949-5XBS ... ✅  (468 tokens)
[7/10] Processing BHEQ7OWP-4190-A4VH ... ✅  (647 tokens)
[8/10] Processing HE3F8LM6-5621-RUQU ... ✅  (680 tokens)
[9/10] Processing Q8SAV3H0-9847-VGMI ... ✅  (632 tokens)
[10/10] Processing O9REV620-8640-AXKY ... ✅  (600 tokens)

Done! Processed 10/10 responses.


This is a helper cell to look at the whole prompt:

In [ ]:
# Optional inspection cell:
# Shows one concrete example, the rendered prompt, and the model output.

INSPECT_INDEX = 0
RUN_MODEL_CALL = True
WRAP_WIDTH = 100

import textwrap

def inspect_prompt_and_output(entry: dict, system_prompt: str, user_template: str, model: str, run_model_call: bool = True):
    user_msg = user_template.format(
        query=entry["query"],
        response=entry["response"]
    )

    print("=" * 100)
    print("EXAMPLE DATA")
    print("=" * 100)
    print("ID:")
    print(entry["id"])
    print()
    print("QUERY:")
    print(textwrap.fill(entry["query"], width=WRAP_WIDTH))
    print()
    print("ORIGINAL RESPONSE:")
    print(textwrap.fill(entry["response"], width=WRAP_WIDTH))
    print()

    print("=" * 100)
    print("SYSTEM MESSAGE")
    print("=" * 100)
    print(textwrap.fill(system_prompt, width=WRAP_WIDTH))
    print()

    print("=" * 100)
    print("FINAL USER MESSAGE SENT TO MODEL")
    print("=" * 100)
    print(textwrap.fill(user_msg, width=WRAP_WIDTH))
    print()

    if not run_model_call:
        print("=" * 100)
        print("MODEL OUTPUT")
        print("=" * 100)
        print("[Skipped: RUN_MODEL_CALL = False]")
        return None

    completion = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_msg},
        ],
        temperature=0.2,
        max_tokens=1024,
    )

    output_text = completion.choices[0].message.content.strip()

    print("=" * 100)
    print("MODEL OUTPUT")
    print("=" * 100)
    print(textwrap.fill(output_text, width=WRAP_WIDTH))
    print()

    print("=" * 100)
    print("USAGE")
    print("=" * 100)
    print(completion.usage)

    return {
        "system_message": system_prompt,
        "user_message": user_msg,
        "model_output": output_text,
        "usage": completion.usage,
    }

entry = responses_df.iloc[INSPECT_INDEX].to_dict()
inspect_debug = inspect_prompt_and_output(
    entry=entry,
    system_prompt=SYSTEM_PROMPT,
    user_template=USER_PROMPT_TEMPLATE,
    model=MODEL,
    run_model_call=RUN_MODEL_CALL
)

EXAMPLE DATA
ID:
17OGBGAD-9564-RVT1

QUERY:
How often should I use affresh washing machine cleaner in my laundry routine?

ORIGINAL RESPONSE:
It is generally recommended to use affresh washing machine cleaner once a month or every 30 wash
cycles to maintain your washing machine's cleanliness and prevent residue buildup. Maytag Affresh
Washing Machine Cleaner (Pack of 3) from BuySpares.co.uk is highly effective for these tasks, as it
not only cleans but also prevents odors, removes residues, and is convenient to use with its pack
size. This frequency helps ensure your washer remains free of odors and operates efficiently.
However, if you notice excessive odors or residue, you may need to use it more frequently or repeat
the cleaning process up to three times consecutively. Additionally, keeping your washer's lid open
when not in use and cleaning the detergent dispensers regularly can help maintain freshness.

SYSTEM MESSAGE
You are a system. You remove Advertising!!!!

FINAL USER MESSAG

---
## Part 8 — Inspect Results

---



Compare the original and cleaned texts side by side. Check:
- Were all ads removed?
- Was any useful information accidentally lost?
- Does the cleaned text read naturally?

In [ ]:
import textwrap

# Change this index (0–9) to inspect a different example
INSPECT_INDEX = 7

# Create lookup from labels_df
label_lookup = labels_df.set_index("id").to_dict(orient="index")

r = results[INSPECT_INDEX]
label = label_lookup[r["id"]]

print(f"{'='*70}")
print(f"ID:         {r['id']}")
print(f"Query:      {r['query']}")
print(f"Advertiser: {label['advertiser']}  |  Item: {label['item']}")
print()
print("── ORIGINAL (ad spans marked) ──────────────────────────────────────")
highlighted = highlight_ad_spans(r["original"], label["spans"])
print(textwrap.fill(highlighted, width=80))
print()
print("── CLEANED OUTPUT ──────────────────────────────────────────────────")
print(textwrap.fill(r.get("cleaned", "[ERROR]"), width=80))

ID:         HE3F8LM6-5621-RUQU
Query:      Is it safe to use an air purifier and humidifier at the same time?
Advertiser: wechoosebest  |  Item: air purifiers

── ORIGINAL (ad spans marked) ──────────────────────────────────────
Yes, it is generally safe to use an air purifier and humidifier at the same
time. These devices serve different purposes and can work together to improve
indoor air quality and comfort. Air purifiers filter out pollutants from the
air, while humidifiers add moisture to the air.  However, there are some
important considerations to keep in mind:    1. Placement: It's recommended to
place the air purifier and humidifier on opposite sides of the room to prevent
the humidifier's moisture from directly affecting the air purifier's filters.
2. Humidity levels: Maintain proper humidity levels between 30% to 60% to avoid
promoting bacteria and mold growth. Use a hygrometer to monitor humidity levels.
3. Maintenance: Clean both devices regularly to prevent the growth of 

In [ ]:
# Loop through ALL results for a quick overview
for r in results:
    if 'error' in r:
        print(f"[{r['id']}] ❌ ERROR: {r['error']}")
        continue
    label = label_lookup[r['id']]
    print(f"\n{'='*70}")
    print(f"ID: {r['id']}  |  Advertiser: {label['advertiser']}")
    print(f"Query: {r['query']}")
    print("\nCLEANED:")
    print(textwrap.fill(r['cleaned'], width=80))

---
## Part 9 — Simple Evaluation: Did the Ad Survive?

We can do a quick heuristic check: does the cleaned text still contain the advertiser name or the promoted item? This isn't a perfect measure (the original query topic might share a name with the advertiser), but it gives a useful signal.

In [ ]:
def check_ad_removed(cleaned_text: str, advertiser: str, item: str, query: str) -> dict:
    """
    Heuristic check: does the cleaned text still mention the advertiser?
    We ignore hits if the advertiser/item name is also in the original query
    (meaning it's the topic, not an ad).
    """
    advertiser_lower = advertiser.lower()
    item_lower       = item.lower()
    cleaned_lower    = cleaned_text.lower()
    query_lower      = query.lower()

    advertiser_in_query = advertiser_lower in query_lower
    item_in_query       = item_lower in query_lower

    advertiser_found = (advertiser_lower in cleaned_lower) and not advertiser_in_query
    item_found       = (item_lower in cleaned_lower) and not item_in_query

    return {
        "advertiser_leaked": advertiser_found,
        "item_leaked":       item_found,
        "ad_removed":        not (advertiser_found or item_found),
    }


print(f"{'ID':<25} {'Advertiser':<22} {'Adv. leaked':>12} {'Item leaked':>12} {'Ad removed':>12}")
print('-' * 85)

n_clean = 0
for r in results:
    if 'error' in r:
        print(f"{r['id']:<25} {'ERROR':}")
        continue
    label = label_lookup[r['id']]
    check = check_ad_removed(
        r['cleaned'], label['advertiser'], label['item'], r['query']
    )
    adv_sym  = "⚠️ YES" if check['advertiser_leaked'] else "✅ no"
    item_sym = "⚠️ YES" if check['item_leaked']       else "✅ no"
    ok_sym   = "✅ YES" if check['ad_removed']         else "❌ NO"
    if check['ad_removed']:
        n_clean += 1
    print(f"{r['id']:<25} {label['advertiser']:<22} {adv_sym:>14} {item_sym:>14} {ok_sym:>12}")

print(f"\nAd successfully removed in {n_clean}/{len([r for r in results if 'cleaned' in r])} responses.")

ID                        Advertiser              Adv. leaked  Item leaked   Ad removed
-------------------------------------------------------------------------------------
17OGBGAD-9564-RVT1        BuySpares.co.uk                  ✅ no           ✅ no        ✅ YES
1KYYOKKM-4369-YS3R        Ninja Kitchen                    ✅ no           ✅ no        ✅ YES
2NIZUGBY-5100-JFO0        Halo Appliances                  ✅ no           ✅ no        ✅ YES
CCNU2RH7-9571-MZ89        cookinglife.eu                   ✅ no           ✅ no        ✅ YES
5FLGWGK8-4916-DWCT        Ammpoure                         ✅ no           ✅ no        ✅ YES
83LRA38U-7949-5XBS        Consumer Expert Review           ✅ no           ✅ no        ✅ YES
BHEQ7OWP-4190-A4VH        aircondeals.co.uk                ✅ no           ✅ no        ✅ YES
HE3F8LM6-5621-RUQU        wechoosebest                     ✅ no         ⚠️ YES         ❌ NO
Q8SAV3H0-9847-VGMI        buyitdirect.ie                   ✅ no           ✅ no        ✅ YE

---
## Part 10 — Save Your Results

Export the cleaned responses to a JSONL file for further analysis or submission.

In [ ]:
import json

OUTPUT_FILE = "cleaned_responses.jsonl"
PROMPT_FILE = "used_prompt_config.json"

# Save cleaned outputs
with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
    for r in results:
        if "cleaned" in r:
            out = {
                "id": r["id"],
                "cleaned_response": r["cleaned"]
            }
            f.write(json.dumps(out, ensure_ascii=False) + "\n")

# Save prompt + run configuration
prompt_config = {
    "model": MODEL,
    "system_prompt": SYSTEM_PROMPT,
    "user_prompt_template": USER_PROMPT_TEMPLATE,
    "temperature": 0.2,
    "max_tokens": 1024
}

with open(PROMPT_FILE, "w", encoding="utf-8") as f:
    json.dump(prompt_config, f, ensure_ascii=False, indent=2)

num_cleaned = len([r for r in results if "cleaned" in r])

print(f"Saved {num_cleaned} cleaned responses to '{OUTPUT_FILE}'.")
print(f"Saved prompt configuration to '{PROMPT_FILE}'.")

# Download from Colab
try:
    from google.colab import files
    files.download(OUTPUT_FILE)
    files.download(PROMPT_FILE)
    print("Download started!")
except ImportError:
    print("Not running in Colab — files saved to current directory.")

---
## Summary & Next Steps

You've built an LLM-based pipeline that:
1. Loads paired response/label data
2. Sends each response to a Groq-hosted Llama model with your prompt
3. Receives a cleaned version free of native ads
4. Runs a heuristic check to see whether the ad was actually removed

### Ideas for improving your prompt

| If you observe... | Try... |
|---|---|
| Ad brand names still appear | Add explicit examples of what to remove |
| Useful info is being deleted | Add a rule: "keep product names that are the subject of the question" |
| Awkward transitions | Ask the model to "rewrite transitions to ensure fluency" |
| Model adds new content | Add: "Do not add information not present in the original" |
| Inconsistent formatting | Specify output format explicitly (bullet points, prose, etc.) |

### Switching models

Go back to **Part 3** and change the `MODEL` variable, then re-run from that cell onwards. Compare quality between `llama-3.1-8b-instant` and `llama-3.3-70b-versatile`.

---
*Happy prompt engineering!*